Step 1 — Airflow install

In [ ]:
pip install apache-airflow

Step 2: airflow db migrate

In [ ]:
!airflow db migrate

2026-10-05T16:48:10.246880Z [info     ] Performing upgrade to the metadata database [airflow.cli.commands.db_command] loc=db_command.py:134 url=sqlite:////root/airflow/airflow.db
2026-10-05T16:48:11.062667Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-10-05T16:48:11.062952Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-10-05T16:48:12.443660Z [info     ] Context impl SQLiteImpl.       [alembic.runtime.migration] loc=migration.py:205
2026-10-05T16:48:12.443925Z [info     ] Will assume non-transactional DDL. [alembic.runtime.migration] loc=migration.py:208
2026-10-05T16:48:12.444482Z [info     ] Creating Airflow database tables from the ORM [airflow.utils.db] loc=db.py:749
2026-10-05T16:48:12.444675Z [info     ] Creating global lock context   [airflow.utils.db] loc=db.py:730
2026-10-05T16:48:12.444854Z [info     ] Pool status: Pool size: 5  Connections in pool: 0 Current Overflow: -4

Step 3 — DAG folder

In [ ]:
mkdir airflow_dags

In [ ]:
cd airflow_dags

/content/airflow_dags


Step 4 — data_extraction.py

In [ ]:
print("Data extraction completed successfully")

Data extraction completed successfully


Step 5 — data_pipeline_dag.py

In [ ]:
from datetime import datetime, timedelta

from airflow import DAG
from airflow.operators.bash import BashOperator
from airflow.operators.empty import EmptyOperator

default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "email_on_retry": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

with DAG(
    "university_etl_orchestration",
    default_args=default_args,
    description="API and Flat File ETL orchestration",
    schedule=timedelta(days=1),
    catchup=False,
) as dag:

    start_pipeline = EmptyOperator(
        task_id="start_pipeline"
    )

    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command="python data_extraction.py",
    )

    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command='echo "ETL Execution completed successfully"',
    )

    start_pipeline >> execute_extraction >> pipeline_complete

2026-10-05T16:52:02.407352Z [warning  ] The `airflow.operators.bash.BashOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.bash.BashOperator'`. [py.warnings] category=DeprecatedImportWarning filename=/tmp/ipykernel_3370/2042307045.py lineno=4
2026-10-05T16:52:02.412996Z [warning  ] The `airflow.operators.empty.EmptyOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.empty.EmptyOperator'`. [py.warnings] category=DeprecatedImportWarning filename=/tmp/ipykernel_3370/2042307045.py lineno=5


Step 6 — DAG file Airflow च्या DAGs folder

In [ ]:
!airflow config get-value core dags_folder

/root/airflow/dags


Step 7 — DAG check

In [ ]:
!airflow dags list

No data found


Step 8 — DAG structure check

In [ ]:
!airflow tasks list university_etl_orchestration

Traceback (most recent call last):
  File "/usr/local/bin/airflow", line 8, in <module>
    sys.exit(main())
             ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/airflow/__main__.py", line 61, in main
    args.func(args)
    ~~~~~~~~~^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/airflow/cli/cli_config.py", line 49, in command
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/airflow/utils/cli.py", line 113, in wrapper
    return f(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/airflow/utils/cli.py", line 464, in _wrapper
    f(*args, **kwargs)
    ~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/airflow/utils/providers_configuration_loader.py", line 54, in wrapped_function
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/airflow/cli/commands/task_command.py", line 315, in task_list
    dag = dag or get_bagged_dag(args.bundle_name, args.dag_id)
                 ~~~~

Step 9 — DAG run

In [ ]:
!airflow dags trigger university_etl_orchestration

Traceback (most recent call last):
  File "/usr/local/bin/airflow", line 8, in <module>
    sys.exit(main())
             ~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/airflow/__main__.py", line 61, in main
    args.func(args)
    ~~~~~~~~~^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/airflow/cli/cli_config.py", line 49, in command
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/airflow/utils/cli.py", line 113, in wrapper
    return f(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/airflow/utils/providers_configuration_loader.py", line 54, in wrapped_function
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/airflow/cli/commands/dag_command.py", line 95, in dag_trigger
    message = api_client.trigger_dag(
        dag_id=args.dag_id,
    ...<4 lines>...
        replace_microseconds=args.replace_microseconds,
    )
  File "/usr/local/lib/python3.13/dist-packages/airflow/api/client/local_cl

In [ ]:
# 1. Create the Airflow DAGs folder
!mkdir -p /root/airflow/dags

# 2. Write the extraction script to disk
with open('/root/airflow/dags/data_extraction.py', 'w') as f:
    f.write('print("Data extraction completed successfully")\n')

# 3. Write the DAG definition to the Airflow DAGs folder
with open('/root/airflow/dags/data_pipeline_dag.py', 'w') as f:
    f.write('''from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.bash import BashOperator
from airflow.providers.standard.operators.empty import EmptyOperator

default_args = {
    "owner": "data_engineering_lab",
    "depends_on_past": False,
    "start_date": datetime(2026, 1, 1),
    "email_on_failure": False,
    "email_on_retry": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=5),
}

with DAG(
    "university_etl_orchestration",
    default_args=default_args,
    description="API and Flat File ETL orchestration",
    schedule=timedelta(days=1),
    catchup=False,
) as dag:

    start_pipeline = EmptyOperator(task_id="start_pipeline")

    execute_extraction = BashOperator(
        task_id="run_extraction_script",
        bash_command="python /root/airflow/dags/data_extraction.py",
    )

    pipeline_complete = BashOperator(
        task_id="log_pipeline_success",
        bash_command=\'echo "ETL Execution completed successfully"\',
    )

    start_pipeline >> execute_extraction >> pipeline_complete
''')

print("DAG and Extraction script successfully written to /root/airflow/dags/")

DAG and Extraction script successfully written to /root/airflow/dags/


In [ ]:
# 4. Verify that Airflow now recognizes the DAG and tasks
!airflow dags list
!airflow tasks list university_etl_orchestration

              |              |         |           |             | bundle_versio
dag_id        | fileloc      | owners  | is_paused | bundle_name | n            
==============+==============+=========+===========+=============+==============
aggregate_reg | /usr/local/l | airflow | True      | example_dag | None         
ional_sales   | ib/python3.1 |         |           | s           |              
              | 3/dist-packa |         |           |             |              
              | ges/airflow/ |         |           |             |              
              | example_dags |         |           |             |              
              | /example_ass |         |           |             |              
              | et_partition |         |           |             |              
              | .py          |         |           |             |              
asset1_produc | /usr/local/l | airflow | True      | example_dag | None         
er            | ib/python3.1

Result:
Apache Airflow was successfully configured and a daily ETL DAG was created with sequential tasks for pipeline initialization, data extraction, and successful pipeline completion logging.